In [1]:
import os

import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

In [2]:
#Some constants/Hyperparameters
CONTEXT_WINDOW = 128
BATCH_SIZE = 32
D_MODEL = 256

In [3]:
#Raw data
train_raw = open('data/ptb.train.txt', 'r').read()
test_raw = open('data/ptb.test.txt', 'r').read()
val_raw = open('data/ptb.val.txt', 'r').read()

In [4]:
#This is the tokenization cell
#TrainTokenizer uses HuggingFaces tokenizers library to create it's own token/vocab set because or data set is
#   rather small with only ~890,000 words
def TrainTokenizer(raw_data, save_name):
    unique_words = len(set(raw_data.split()))
    token_count = int(unique_words * (4/3))

    tokenizer = Tokenizer(BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=token_count, 
        special_tokens=["<unk>", "<s>", "</s>"]
    )

    temp_file = "temp_train_data.txt"
    with open(temp_file, "w", encoding="utf-8") as f:
        f.write(raw_data)
    
    tokenizer.train([temp_file], trainer)

    os.remove(temp_file)

    os.makedirs("data/tokenized", exist_ok=True)
    save_path = f"data/tokenized/{save_name}.json"
    tokenizer.save(save_path)

    print(f"Created tokenized dataset with {token_count} tokens and saved to {save_path}")
    return tokenizer

#This uses the tokenizer we trained and actually tokenized the dataset so that now it's just a long list of
#   numbers/tokens 
def Tokenize(raw_data, save_name):
    my_tokenizer = TrainTokenizer(raw_data, save_name)
    tokenized_data = my_tokenizer.encode(raw_data).ids
    print("Tokenized all data")
    return tokenized_data

In [5]:
#Actually tokenizing the dataset
train_tokens = Tokenize(train_raw, "ptb.train_tokens.txt")

#Just checking if the tokenization looks good
print(train_tokens[:10])
print(train_raw[:100])

Created tokenized dataset with 13332 tokens and saved to data/tokenized/ptb.train_tokens.txt.json
Tokenized all data
[8119, 277, 2191, 246, 879, 50, 384, 320, 111, 232]
 aer banknote berlitz calloway centrust cluett fromstein gitano guterman hydro-quebec ipo kia memote


In [6]:
#This cell is where we create our training batches
#We use Pytorch dataset to act as a plucking tool to pluck out a single training/target
#   tensors from the dataset
class PTBTrain(Dataset):
    def __init__(self, token_list, context_window):
        self.token_list = token_list
        self.context_window = context_window
    
    def __len__(self):
        return len(self.token_list) - self.context_window

    def __getitem__(self, idx):
        x = torch.tensor(self.token_list[idx:idx + self.context_window], dtype=torch.long)
        y = torch.tensor(self.token_list[idx + 1:idx + self.context_window + 1], dtype=torch.long)

        return x, y

#Creating dataset instance
train_data = PTBTrain(train_tokens, CONTEXT_WINDOW)

#Pytorches dataloader abstracts all the logic underneath and actually creates the mini_batches we will be
#   training and calculating the loss on
#Handles all shuffling between epochs
training_batches = DataLoader(
    dataset=train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

In [7]:
#Just checking if the dataloader works and gives the correct matrix and it does
#Dataloader pre sets the suffled indeces and then uses the dataset class to get those
#   indices it needs, but it only holds one batch in memory at once and plucks the next
#   index once prompted to
for batch_x, batch_y in training_batches:
    #Shape: (batch_size, context_window)
    print(batch_x.shape)
    print(batch_y.shape)
    break

torch.Size([32, 128])
torch.Size([32, 128])


In [8]:
#This is the embedding layer and what actually gets fed into the model
#This is the simpler version for now with just regular plain embeddings
#   but I will soon add the RoPE math and everything
class RoPEEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model)
    
    def forward(self, batch):
        return self.token_embedding(batch)

In [9]:
#Creating embedidng layer instance
embedding_layer = RoPEEmbeddings(13332, D_MODEL)

#Checking if batch got embedded correctly
for batch_x, batch_y in training_batches:
    #Shape: (batch_size, coontext_window, d_model)
    print(embedding_layer.forward(batch_x).shape)
    break

torch.Size([32, 128, 256])


In [ ]:
#Creating the actual Transfomer Architecture now
class MultiHeadedAttention(nn.Module):
    pass

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.GeLU = nn.GELU()
    
    def forward(self, batch):
        return self.w2(self.GeLU(self.w1(batch)))